In [ ]:
import torch
from PIL import Image
import pandas as pd
from pathlib import Path
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)

In [ ]:
# -------------------------
# CONFIG
# -------------------------
MODEL_NAME = "google/medgemma-27b-it"        # 27B multimodal instruction-tuned
IMAGE_DIR = "./data/LLaVA-Med/images/ADR_images"                   # folder with your images
OUTPUT_CSV = "./data/LLaVA-Med/MedGemma_Localization_D1500_06Dec2025_V1.csv"

# Your task prompt
BASE_PROMPT = (
    """You are a clinical diagnostic assistant. Analyze the provided image for visual evidence of an Adverse Drug Effect (ADE).
If an ADE is detected, provide the location of the ADE."""
)


In [ ]:
# -------------------------
# QUANTIZATION CONFIG (4-bit)
# -------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
# -------------------------
# LOAD MODEL & PROCESSOR (QUANTIZED)
# -------------------------
print("Loading quantized MedGemma 27B...")
token = os.environ["HF_TOKEN"]   # <-- paste your HF token here

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    use_auth_token=token
)
# processor = AutoProcessor.from_pretrained(MODEL_NAME)

# device = model.device
# print(f"Model loaded on device: {device}")


In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)

# Optional: force eager attention if you want to avoid SDPA kernels
if hasattr(model, "config"):
    if hasattr(model.config, "attn_implementation"):
        model.config.attn_implementation = "eager"
    if hasattr(model.config, "_attn_implementation"):
        model.config._attn_implementation = "eager"

# Choose a primary device for inputs
if hasattr(model, "hf_device_map") and isinstance(model.hf_device_map, dict):
    dev_names = [d for d in model.hf_device_map.values() if d != "meta"]
    device = torch.device(dev_names[0]) if dev_names else torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using main device: {device}")

In [ ]:
# -------------------------
# HELPER: RUN MEDGEMMA ON ONE IMAGE
# -------------------------
def run_medgemma_on_image(image_path: Path, base_prompt: str) -> str:
    """
    Run MedGemma 27B (image-text-to-text) on a single image with a given prompt.
    Returns generated text.
    """
    image = Image.open(image_path).convert("RGB")

    # MedGemma expects <start_of_image> token in the prompt for vision input.
    prompt = f"<start_of_image> {base_prompt}"

    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt",
    )

    # move inputs to the main device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
        )
        # Drop the prompt tokens so we only decode the new generation
        generation = generation[0, input_len:]

    decoded = processor.decode(generation, skip_special_tokens=True)
    return decoded.strip()


In [ ]:
# -------------------------
# MAIN LOOP
# -------------------------
def main():
    image_dir = Path(IMAGE_DIR)
    if not image_dir.exists():
        raise FileNotFoundError(f"Image directory not found: {image_dir.resolve()}")

    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    image_files = [p for p in image_dir.iterdir() if p.suffix.lower() in exts]

    if not image_files:
        raise RuntimeError(f"No image files found in {image_dir.resolve()}")

    print(f"Found {len(image_files)} images in {image_dir}")

    rows = []
    add_prompt = True  # only first row will carry the prompt

    i = 0
    top = 10 #####

    print(f"Prompt: {BASE_PROMPT}")

    for img_path in sorted(image_files):
        i = i + 1
        if(i > top):
            break
        
        print(f"Processing: {img_path.name}")

        try:
            response = run_medgemma_on_image(img_path, BASE_PROMPT)
        except Exception as e:
            print(f"⚠️ Error processing {img_path.name}: {e}")
            response = f"ERROR: {e}"

        if add_prompt:
            row_prompt = BASE_PROMPT
            add_prompt = False
        else:
            row_prompt = ""  # empty for all other rows

        print(f"Response: {response}")

        rows.append(
            {
                "filename": img_path.name,
                "prompt": row_prompt,
                "response": response,
            }
        )

    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print(f"\n✅ Saved {len(rows)} rows to {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

In [ ]:
import os
from pathlib import Path

import torch
from PIL import Image
import pandas as pd
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)

# ------------------------------------------------
# CONFIG
# ------------------------------------------------
MODEL_NAME = "google/medgemma-27b-it"   # 27B multimodal, instruction-tuned

IMAGE_DIR = "./data/LLaVA-Med/images/ADR_images"
OUTPUT_CSV = (
    "./data/LLaVA-Med/"
    "MedGemma_Localization_D1500_06Dec2025_V4.csv"
)

# Instruction for EACH IMAGE
BASE_PROMPT = (
    "Carefully examine this medical image and describe all relevant findings, "
    "including the location of any adverse drug event (ADE) on the body. "
    "Answer in one or two concise sentences."
)

# ------------------------------------------------
# QUANTIZATION CONFIG (4-bit for 27B)
# ------------------------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# ------------------------------------------------
# LOAD MODEL & PROCESSOR (DIRECT, NO PIPELINE)
# ------------------------------------------------
print("Loading quantized MedGemma 27B...")

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    attn_implementation="eager",   # with torch>=2.6 this is fine
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(MODEL_NAME)

if hasattr(processor, "tokenizer"):
    processor.tokenizer.padding_side = "right"

device = model.device
print(f"Model main device: {device}")

# ------------------------------------------------
# HELPER: RUN MEDGEMMA ON ONE IMAGE
# ------------------------------------------------
def run_medgemma_on_image(image_path: Path, base_prompt: str) -> str:
    """
    Run MedGemma 27B on a single image with a given textual instruction,
    following the official 'run the model directly' pattern.
    """
    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "You are an expert medical vision-language assistant. "
                        "Always provide concise, clinically meaningful descriptions."
                    ),
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": base_prompt},
                {"type": "image", "image": image},
            ],
        },
    ]

    # EXACT pattern from model card, but with our messages & quantized model
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
        )
        # Keep only newly generated tokens
        generation = generation[0][input_len:]

    decoded = processor.decode(generation, skip_special_tokens=True).strip()

    # Optional debug: uncomment if you want to see raw outputs in the console
    print(f"[DEBUG] {image_path.name} -> {repr(decoded)}")

    if not decoded:
        decoded = "[EMPTY_OUTPUT]"
    return decoded

# ------------------------------------------------
# MAIN LOOP OVER ALL IMAGES
# ------------------------------------------------
def main():
    image_dir = Path(IMAGE_DIR)
    if not image_dir.exists():
        raise FileNotFoundError(f"Image directory not found: {image_dir.resolve()}")

    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    image_files = [p for p in image_dir.iterdir() if p.suffix.lower() in exts]

    if not image_files:
        raise RuntimeError(f"No image files found in {image_dir.resolve()}")

    print(f"Found {len(image_files)} images in {image_dir}")

    rows = []
    add_prompt = True  # only first row will contain the prompt

    for img_path in sorted(image_files):
        print(f"Processing: {img_path.name}")

        try:
            response = run_medgemma_on_image(img_path, BASE_PROMPT)
        except Exception as e:
            print(f"⚠️ Error processing {img_path.name}: {e}")
            response = f"ERROR: {e}"

        if add_prompt:
            row_prompt = BASE_PROMPT
            add_prompt = False
        else:
            row_prompt = ""  # blank for all subsequent rows

        print(f"Response: {response}")

        rows.append(
            {
                "filename": img_path.name,
                "prompt": row_prompt,
                "response": response,
            }
        )

    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print(f"\n✅ Saved {len(rows)} rows to {OUTPUT_CSV}")


if __name__ == "__main__":
    main()


In [ ]:
import torch

model.eval()

prompt = "You are a medical assistant. Answer briefly.\nPatient: I have a mild headache. What could I do?"
inputs = processor(
    text=prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False
    )

# Some processors return a list, handle both
if hasattr(processor, "batch_decode"):
    response = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
else:
    response = model.config.tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("=== MODEL RESPONSE ===")
print(response)


In [ ]:
import torch

model.eval()

# 1) Build messages in the official MedGemma format
messages = [
    {
        "role": "system",
        "content": [
            {"type": "text", "text": "You are a helpful medical assistant."}
        ],
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "How do you differentiate bacterial from viral pneumonia? Answer briefly."}
        ],
    },
]

# 2) Use the chat template to create input_ids
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

# Move to same device as your model
inputs = {k: v.to(device) for k, v in inputs.items()}

input_len = inputs["input_ids"].shape[-1]

# 3) Generate
with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
    )

# 4) Drop the prompt tokens and decode only the new tokens
generated = output_ids[0, input_len:]
decoded = processor.decode(generated, skip_special_tokens=True)
print("output_ids shape:", output_ids.shape)
print("last 20 token ids:", output_ids[0, -20:])


print("=== MODEL RESPONSE ===")
print(repr(decoded))  # repr() so you can see if it's actually empty or just whitespace


In [ ]:
import torch

model.eval()

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful medical assistant."}],
    },
    {
        "role": "user",
        "content": [{"type": "text", "text": "How do you differentiate bacterial from viral pneumonia?"}],
    },
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

print("input_ids shape:", inputs["input_ids"].shape)
input_len = inputs["input_ids"].shape[-1]
print("input_len:", input_len)

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
    )

print("output_ids shape:", output_ids.shape)

# Show *new* tokens only
new_tokens = output_ids[0, input_len:]
print("new_tokens shape:", new_tokens.shape)
print("first 20 new token ids:", new_tokens[:20])

decoded_all = processor.decode(output_ids[0], skip_special_tokens=True)
decoded_new = processor.decode(new_tokens, skip_special_tokens=True)

print("=== DECODED ALL (repr) ===")
print(repr(decoded_all))
print("=== DECODED NEW (repr) ===")
print(repr(decoded_new))
